# GitLab Projects Export
Exports all projects under a GitLab group to **XLSX + JSON** with:
- `team_name`, `package_json`, `int_ext`, `component`
- `import_filename`, `import_file_url`, `import_statement` (scanned from .js/.ts/.jsx/.tsx)

**Performance**: projects and file fetches run in parallel via `ThreadPoolExecutor`.

In [ ]:
import getpass
import json
import logging
import re
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Any, Dict, List, Optional, Tuple

import gitlab
import pandas as pd

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
logger = logging.getLogger('gitlab_export')
logger.info('Imports loaded.')

## Configuration

In [ ]:
GITLAB_URL      = 'https://devcloud.ubs.net'
GROUP_PATH      = 'ubs/gwma'
OUTPUT_XLSX     = 'gitlab_projects_export.xlsx'
OUTPUT_JSON     = 'gitlab_projects_export.json'
PER_PAGE        = 100
MAX_WORKERS     = 10   # parallel project threads
FILE_WORKERS    = 5    # parallel file-fetch threads per project

SOURCE_EXTENSIONS = ('.jsx', '.tsx', '.js', '.ts')

# Matches: import { ... } from '@ubs.websdk/...' or '@uwr.../...'
IMPORT_PATTERN = re.compile(
    r'^[ \t]*import\s+\{[^}]+\}(?:\s*,\s*\{[^}]+\})*\s+from\s+[\'"](@ ubs\.websdk/[^\'"]+|@uwr[^\'"]+)[\'"].*$',
    re.MULTILINE,
)

logger.info('Configuration set.')

## Authentication

In [ ]:
private_token = getpass.getpass('Enter your GitLab private token: ')
client = gitlab.Gitlab(GITLAB_URL, private_token=private_token)
logger.info('GitLab client ready for %s', GITLAB_URL)

## Helper Functions

In [ ]:
def extract_team_name(web_url: str) -> str:
    """Second-last URL path segment = team folder.
    https://host/ubs/gwm/xxx/yyy/zzz/projectname -> zzz
    """
    try:
        segments = web_url.rstrip('/').split('//', 1)[-1].split('/')[1:]
        if len(segments) >= 2:
            return segments[-2]
    except Exception:
        pass
    return ''


def get_package_json_info(project: Any, ref: str) -> Tuple[str, str]:
    """Check dependencies in package.json for @uwr/ prefixed packages.
    Returns ('Yes'/'No', 'internal'/'external'/'-').
    """
    try:
        raw = project.files.get(file_path='package.json', ref=ref).decode()
        content = raw if isinstance(raw, str) else raw.decode('utf-8')
        pkg = json.loads(content)
        deps = pkg.get('dependencies', {})
        uwr_deps = [d for d in deps if d.startswith('@uwr/')]
        if uwr_deps:
            logger.info('  [pkg.json] internal | %s | %s', project.path_with_namespace, uwr_deps)
            return 'Yes', 'internal'
        logger.info('  [pkg.json] external | %s', project.path_with_namespace)
        return 'Yes', 'external'
    except gitlab.exceptions.GitlabGetError:
        logger.info('  [pkg.json] not found | %s', project.path_with_namespace)
        return 'No', '-'
    except json.JSONDecodeError as e:
        logger.warning('  [pkg.json] parse error | %s | %s', project.path_with_namespace, e)
        return 'Yes', 'external'
    except Exception as e:
        logger.warning('  [pkg.json] error | %s | %s', project.path_with_namespace, e)
        return 'No', '-'


def _fetch_content(project: Any, path: str, ref: str) -> Optional[str]:
    try:
        raw = project.files.get(file_path=path, ref=ref).decode()
        return raw if isinstance(raw, str) else raw.decode('utf-8', errors='replace')
    except Exception:
        return None


def scan_imports(project: Any, ref: str) -> Tuple[str, str, str]:
    """Walk repo tree, find .js/.ts/.jsx/.tsx files, extract qualifying imports.
    Returns bracket-delimited (import_filename, import_file_url, import_statement).
    """
    try:
        items = project.repository_tree(ref=ref, recursive=True, all=True, per_page=100)
    except Exception as e:
        logger.warning('  [scan] tree error | %s | %s', project.path_with_namespace, e)
        return '', '', ''

    source_files = [
        item for item in items
        if item.get('type') == 'blob' and item['path'].endswith(SOURCE_EXTENSIONS)
    ]
    logger.info('  [scan] %d source files | %s', len(source_files), project.path_with_namespace)

    filenames, file_urls, stmts = [], [], []

    def scan_one(item):
        path = item['path']
        content = _fetch_content(project, path, ref)
        if not content:
            return None
        matches = list(IMPORT_PATTERN.finditer(content))
        if not matches:
            return None
        fname = path.split('/')[-1]
        furl = f"{project.web_url}/-/blob/{ref}/{path}"
        return [(fname, furl, m.group(0).strip()) for m in matches]

    with ThreadPoolExecutor(max_workers=FILE_WORKERS) as pool:
        for result in pool.map(scan_one, source_files):
            if result:
                for fname, furl, stmt in result:
                    filenames.append(fname)
                    file_urls.append(furl)
                    stmts.append(stmt)

    if not filenames:
        logger.info('  [scan] no matching imports | %s', project.path_with_namespace)
        return '', '', ''

    logger.info('  [scan] %d import(s) | %s', len(filenames), project.path_with_namespace)
    fmt = lambda lst: ''.join(f'[{v}]' for v in lst)
    return fmt(filenames), fmt(file_urls), fmt(stmts)


def process_project(project: Any) -> Dict[str, Any]:
    namespace = project.namespace or {}
    web_url = project.web_url
    ref = project.default_branch or 'main'
    team_name = extract_team_name(web_url)
    package_json, int_ext = get_package_json_info(project, ref)
    import_filename, import_file_url, import_statement = scan_imports(project, ref)
    return {
        'project_id':          project.id,
        'name':                project.name,
        'path':                project.path,
        'path_with_namespace': project.path_with_namespace,
        'group_path':          namespace.get('full_path'),
        'web_url':             web_url,
        'description':         project.description,
        'visibility':          project.visibility,
        'archived':            project.archived,
        'created_at':          project.created_at,
        'last_activity_at':    project.last_activity_at,
        'updated_at':          project.updated_at,
        'default_branch':      ref,
        'forks_count':         getattr(project, 'forks_count', None),
        'star_count':          getattr(project, 'star_count', None),
        'open_issues_count':   getattr(project, 'open_issues_count', None),
        'team_name':           team_name,
        'package_json':        package_json,
        'int_ext':             int_ext,
        'component':           '',
        'import_filename':     import_filename,
        'import_file_url':     import_file_url,
        'import_statement':    import_statement,
    }


logger.info('All helper functions defined.')

## Fetch & Process Projects (Parallel)

In [ ]:
logger.info("Fetching group '%s'", GROUP_PATH)
group = client.groups.get(GROUP_PATH)
proj_refs = group.projects.list(include_subgroups=True, all=True, per_page=PER_PAGE)
total = len(proj_refs)
logger.info('Found %d project references — processing with %d workers', total, MAX_WORKERS)

rows: List[Dict[str, Any]] = []
done = 0

def fetch_and_process(proj_ref, idx):
    logger.info('[%d/%d] → %s', idx, total, proj_ref.path_with_namespace)
    project = client.projects.get(proj_ref.id)
    return process_project(project)

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    future_map = {
        pool.submit(fetch_and_process, ref, idx): ref
        for idx, ref in enumerate(proj_refs, start=1)
    }
    for future in as_completed(future_map):
        done += 1
        ref = future_map[future]
        try:
            row = future.result()
            rows.append(row)
            logger.info('[%d/%d ✓] %s', done, total, row['path_with_namespace'])
        except Exception as e:
            logger.error('[%d/%d ✗] %s | %s', done, total, ref.path_with_namespace, e)

logger.info('Done: %d/%d projects processed', len(rows), total)

## Preview Data

In [ ]:
df = pd.DataFrame(rows)

for col in ('created_at', 'last_activity_at', 'updated_at'):
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')
        if pd.api.types.is_datetime64tz_dtype(df[col]):
            df[col] = df[col].dt.tz_localize(None)

df.sort_values(by=['group_path', 'name'], inplace=True, ignore_index=True)

print(f'Total projects: {len(df)}')
print(f'With imports found: {(df["import_filename"] != "").sum()}')
df[['name', 'team_name', 'package_json', 'int_ext', 'import_filename', 'import_statement']].head(10)

## Write XLSX

In [ ]:
if rows:
    logger.info('Writing XLSX → %s', OUTPUT_XLSX)
    with pd.ExcelWriter(OUTPUT_XLSX, engine='openpyxl') as writer:
        df.to_excel(writer, index=False, sheet_name='projects')
        ws = writer.sheets['projects']
        for col_cells in ws.columns:
            max_len = max((len(str(c.value)) if c.value is not None else 0) for c in col_cells)
            ws.column_dimensions[col_cells[0].column_letter].width = min(max_len + 4, 80)
    logger.info('XLSX done: %s', OUTPUT_XLSX)
    print(f'✅ XLSX saved: {OUTPUT_XLSX}')
else:
    logger.warning('No rows to write.')

## Write JSON

In [ ]:
if rows:
    logger.info('Writing JSON → %s', OUTPUT_JSON)
    with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
        json.dump(rows, f, indent=2, default=str)
    logger.info('JSON done: %s', OUTPUT_JSON)
    print(f'✅ JSON saved: {OUTPUT_JSON}')
else:
    logger.warning('No rows to write.')